# Capa Silver - Catálogo de Clientes

In [1]:
from spark_session import *
from transformation_functions import *
from delta.tables import DeltaTable
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, max, col

## SparkSession

In [2]:
spark = create_spark_session(
    "retail-dlh-silver",
    configs=[
        configure_delta_extensions,
        configure_delta_catalog,
        configure_s3
    ]
)

## Extracción

In [3]:
# - Obtiene dataframe de la capa bronze
df_customer = spark.table("bronze.customer")

df_customer = df_customer \
    .drop(col("DEALERSHIP_ID")) \
    .drop(col("REGION")) \
    .drop(col("CUST_DATE"))

df_customer = df_customer.withColumn("INCOME", col("INCOME").cast("decimal(9, 0)"))

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


## Transformaciones

In [4]:
# - Filtra los clientes validos
df_customer_filter = df_customer.where(
    (df_customer["CUSTOMER_NO"].isNotNull()) & (df_customer["CUSTOMER_NO"] != 99999)
)

# - Asignación de consecutivo por cliente
df_consecutivo = df_customer_filter.withColumn("CONSECUTIVO", row_number().over(
        Window.partitionBy(col("CUSTOMER_NO")).orderBy(col("CUSTOMER_NO"))
    )
)

# - Obtiene el maximo consecutivo por cliente
df_customer_max = df_consecutivo.groupBy("CUSTOMER_NO").agg(
    max(col("CONSECUTIVO")).alias("MAX_CONSECUTIVO")
).withColumnRenamed("CUSTOMER_NO", "MAX_CUSTOMER_NO")

# - Obtiene el cliente unico, nombre completo, formato del telefono, genero completo, grupo de edad
df_join_customer = df_consecutivo.join(
    df_customer_max \
    , ((df_consecutivo.CUSTOMER_NO == df_customer_max.MAX_CUSTOMER_NO) &
       (df_consecutivo.CONSECUTIVO == df_customer_max.MAX_CONSECUTIVO)) \
    , "inner"
).withColumn("CUST_NAME", 
             combinar_nombre_apellido_udf(col("FIRSTNAME"), col("LASTNAME"))
).withColumn("CUST_PHONE_NMBR", 
             formato_telefono_udf(col("PHONE_NUMBER"))             
).withColumn("CUST_GENDER",
             genero_completo_udf(col("GENDER"))
).withColumn("CUST_AGE_GROUP", 
             grupo_edad_udf(col("AGE"))
)

# - Renombra columnas
df_customer_final = df_join_customer.select(
    col("CUSTOMER_NO").alias("CUST_ID"),
    col("CUST_NAME"),
    col("ADDRESS").alias("CUST_ADDRESS"),
    col("CITY").alias("CUST_CITY"),
    col("STATE").alias("CUST_STATE"),
    col("ZIP").alias("CUST_ZIP_CODE"),
    col("COUNTRY").alias("CUST_COUNTRY"),
    col("CUST_PHONE_NMBR"),
    col("CUST_GENDER"),
    col("CUST_AGE_GROUP"),
    col("INCOME").alias("CUST_INCOME"),
    col("EMAIL").alias("CUST_E_MAIL"),
    col("AGE").alias("CUST_AGE"),
)

## Carga

In [5]:
if DeltaTable.isDeltaTable(spark, "silver.customers"):
  ## --Inserta nuevos regitros, actualiza registros con cambios
  deltaTblSilCustomers = DeltaTable.forName(spark, "silver.customers")
  deltaTblSilCustomers.alias('SILV') \
    .merge(
      df_customer_final.alias('BRON'),
      condition='BRON.CUST_ID = SILV.CUST_ID'
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()
else:
  ## --Inserta registros por primera vez
  df_customer_final.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.customers")

## Vacuum

In [6]:
# - Actualiza la configuración para que pueda establecer un período de retención cero.
#spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

#- Ejecutemos el comando de VACUUM 
#spark.sql(f"VACUUM '{vPathSilv}CUSTOMERS.delta' RETAIN 0 HOURS").show(truncate=False)

## Cierre de SparkSession

In [7]:
spark.stop()